# 参考题解：Head-Aware LoRA

实现独立逐头低秩适配与 Domain-Invariant Gating。

核心思路：将输入拆成多个头，每个头使用独立的低秩 A/B，再用归一化门控缩放各头更新，最后拼回模型维度。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionHeadPurificationLoRA(nn.Module):
    def __init__(self, d_model: int, num_heads: int, rank: int, alpha: float = 1.0):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model, self.num_heads = d_model, num_heads
        self.head_dim, self.rank = d_model // num_heads, rank
        self.scale = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(num_heads,self.head_dim,rank) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(num_heads,rank,self.head_dim))
        self.head_gates = nn.Parameter(torch.ones(num_heads))
        self.weight = nn.Parameter(torch.randn(d_model,d_model) * 0.02, requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b,s,_=x.shape
        base=x@self.weight
        xh=x.view(b,s,self.num_heads,self.head_dim).transpose(1,2)
        update=(xh@self.lora_A@self.lora_B)*self.scale
        gates=F.softmax(self.head_gates,0).view(1,self.num_heads,1,1)*self.num_heads
        update=(update*gates).transpose(1,2).contiguous().view(b,s,self.d_model)
        return base+update
